In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


In [2]:
df = pd.read_csv("/content/Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [3]:
y = df["Survived"]

In [4]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]

X = df[features].copy()


In [5]:
categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
]

numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]


In [6]:
for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])

for col in ['Age', 'Fare']: # Impute specific numerical columns
    if col in X.columns:
        X[col] = X[col].fillna(X[col].median())

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

In [9]:
l1_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l1_model.fit(X_train, y_train)

y_pred_l1 = l1_model.predict(X_test)

y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]

In [10]:
#logistic_model.fit(X_train, y_train)

In [11]:
#y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1]

In [12]:
accuracy = accuracy_score(y_test, y_pred_l1)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l1,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l1), 4))


BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.8045

Confusion Matrix:
[[98 12]
 [23 46]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.81      0.89      0.85       110
      Placed       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


ROC-AUC:
0.8457
